# Project 1500 — Opening Analysis
**Scope:** Rapid games, April 2025 – present  
**Goal:** Identify which openings to practice to break the 1400 plateau

In [ ]:
import json
import glob
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

USERNAME = "jf4bes"
GAMES_DIR = "games"
MIN_GAMES = 5  # minimum games to show in aggregations

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 60)

## Load rapid games since April 2025

In [ ]:
EXCLUDE = {"2025_01", "2025_02", "2025_03"}

def load_rapid_games():
    rows = []
    files = sorted(glob.glob(f"{GAMES_DIR}/2025_*.json") + glob.glob(f"{GAMES_DIR}/2026_*.json"))
    files = [f for f in files if not any(ex in f for ex in EXCLUDE)]

    for f in files:
        with open(f, encoding="utf-8") as fh:
            games = json.load(fh)
        for g in games:
            if g.get("time_class") != "rapid":
                continue

            white = g.get("white", {})
            black = g.get("black", {})

            if white.get("username", "").lower() == USERNAME.lower():
                color = "white"
                my = white
                opp = black
            elif black.get("username", "").lower() == USERNAME.lower():
                color = "black"
                my = black
                opp = white
            else:
                continue

            result = my.get("result", "")
            if result == "win":
                outcome = "win"
            elif result in ("checkmated", "timeout", "resigned", "lose", "abandoned"):
                outcome = "loss"
            elif result in ("agreed", "stalemate", "repetition", "insufficient", "timevsinsufficient", "50move"):
                outcome = "draw"
            else:
                continue

            opening_raw = g.get("opening", "") or ""
            # opening field on chess.com is a URL like https://www.chess.com/openings/Sicilian-Defense-...
            # extract slug from URL
            if "/openings/" in opening_raw:
                slug = opening_raw.split("/openings/")[-1].strip("/")
            else:
                slug = opening_raw

            # Convert slug to human-readable: replace hyphens, split on known patterns
            parts = slug.replace("-", " ").split("  ")
            opening_full = slug.replace("-", " ")

            rows.append({
                "end_time": g.get("end_time", 0),
                "color": color,
                "outcome": outcome,
                "my_rating": my.get("rating", 0),
                "opp_rating": opp.get("rating", 0),
                "end_reason": result,
                "time_control": g.get("time_control", ""),
                "opening_url": opening_raw,
                "opening_full": opening_full,
                "num_moves": len(g.get("pgn", "").split("\n")) if g.get("pgn") else 0,
            })

    df = pd.DataFrame(rows)
    df["date"] = pd.to_datetime(df["end_time"], unit="s")
    df = df.sort_values("end_time").reset_index(drop=True)
    return df

df = load_rapid_games()
print(f"Loaded {len(df)} rapid games")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"\nOutcomes:\n{df['outcome'].value_counts()}")

## Rating progression

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df["date"], df["my_rating"], linewidth=0.8, alpha=0.8, color="steelblue")
ax.axhline(1400, color="red", linestyle="--", linewidth=1, label="1400 target")
ax.axhline(1500, color="green", linestyle="--", linewidth=1, label="1500 goal")
ax.set_title("Rapid rating — April 2025 to present")
ax.set_ylabel("Rating")
ax.legend()
plt.tight_layout()
plt.show()
print(f"Current rating: {df['my_rating'].iloc[-1]}  |  Peak: {df['my_rating'].max()}  |  Low: {df['my_rating'].min()}")

## Build opening levels
The Chess.com opening URL slug is structured: `Main-Opening-Name-Variation-Subvariation`  
We parse it into 3 levels of specificity.

In [ ]:
def parse_opening_levels(url):
    """Extract opening hierarchy from chess.com opening URL."""
    if not url or "/openings/" not in url:
        return "Unknown", "Unknown", "Unknown"

    slug = url.split("/openings/")[-1].strip("/")
    # Chess.com slugs use hyphens within words and double-hyphens or capitalisation breaks between parts
    # Split on uppercase transitions to find variation boundaries
    # e.g. Sicilian-Defense-Najdorf-Variation-English-Attack
    # Known variation keywords that mark level boundaries
    parts = slug.split("-")

    # Find boundary words that typically separate main opening from variation
    boundary_keywords = {
        "Variation", "Defense", "Attack", "Gambit", "Opening",
        "System", "Game", "Indian", "Dutch", "English", "French",
        "Italian", "Spanish", "Sicilian"
    }

    # Strategy: find the last occurrence of a boundary keyword in first 4 tokens
    # to define the main opening name, then subsequent tokens = variation
    human = " ".join(parts)

    # Level 1: everything up to and including first "Defense"/"Attack"/"Gambit"/"Game"
    # Level 2: + next group of words
    # Level 3: full slug
    level_breaks = []
    for i, p in enumerate(parts):
        if p in ("Defense", "Defence", "Attack", "Gambit", "Game", "Opening", "Variation", "System"):
            level_breaks.append(i)

    if len(level_breaks) == 0:
        l1 = human
        l2 = human
        l3 = human
    elif len(level_breaks) == 1:
        l1 = " ".join(parts[:level_breaks[0]+1])
        l2 = human
        l3 = human
    else:
        l1 = " ".join(parts[:level_breaks[0]+1])
        l2 = " ".join(parts[:level_breaks[1]+1])
        l3 = human

    return l1, l2, l3

df[["level1", "level2", "level3"]] = df["opening_url"].apply(
    lambda u: pd.Series(parse_opening_levels(u))
)

print("Sample opening levels:")
df[["level1", "level2", "level3"]].drop_duplicates().head(15)

## Helper: opening stats table

In [ ]:
def opening_table(data, group_col, min_games=MIN_GAMES, color_filter=None):
    if color_filter:
        data = data[data["color"] == color_filter]
    grp = data.groupby(group_col)["outcome"].value_counts().unstack(fill_value=0)
    for col in ["win", "loss", "draw"]:
        if col not in grp:
            grp[col] = 0
    grp["total"] = grp[["win", "loss", "draw"]].sum(axis=1)
    grp = grp[grp["total"] >= min_games].copy()
    grp["win%"] = (grp["win"] / grp["total"] * 100).round(1)
    grp["loss%"] = (grp["loss"] / grp["total"] * 100).round(1)
    return grp[["win", "loss", "draw", "total", "win%", "loss%"]].sort_values("total", ascending=False)

## Level 1 — Main opening family (broadest)

In [ ]:
print("=== AS WHITE ===")
display(opening_table(df, "level1", color_filter="white"))
print("\n=== AS BLACK ===")
display(opening_table(df, "level1", color_filter="black"))

## Level 2 — Opening + first variation

In [ ]:
print("=== AS WHITE ===")
display(opening_table(df, "level2", color_filter="white"))
print("\n=== AS BLACK ===")
display(opening_table(df, "level2", color_filter="black"))

## Level 3 — Full opening line (most specific)

In [ ]:
print("=== AS WHITE ===")
display(opening_table(df, "level3", color_filter="white"))
print("\n=== AS BLACK ===")
display(opening_table(df, "level3", color_filter="black"))

## Worst openings by win rate (Level 2, min 5 games)

In [ ]:
all_l2 = opening_table(df, "level2", min_games=5)
worst = all_l2.sort_values("win%").head(10)

fig, ax = plt.subplots(figsize=(12, 5))
colors = ["tomato" if w < 40 else "orange" if w < 50 else "steelblue" for w in worst["win%"]]
bars = ax.barh(worst.index, worst["win%"], color=colors)
ax.axvline(50, color="gray", linestyle="--", linewidth=1)
ax.set_xlabel("Win %")
ax.set_title("10 worst openings by win rate (min 5 games, all colors)")
for bar, (_, row) in zip(bars, worst.iterrows()):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f"{row['win%']}% ({int(row['total'])} games)", va="center", fontsize=8)
plt.tight_layout()
plt.show()

## Best openings by win rate (Level 2, min 5 games)

In [ ]:
best = all_l2.sort_values("win%", ascending=False).head(10)

fig, ax = plt.subplots(figsize=(12, 5))
colors = ["green" if w >= 60 else "steelblue" for w in best["win%"]]
bars = ax.barh(best.index, best["win%"], color=colors)
ax.axvline(50, color="gray", linestyle="--", linewidth=1)
ax.set_xlabel("Win %")
ax.set_title("10 best openings by win rate (min 5 games, all colors)")
for bar, (_, row) in zip(bars, best.iterrows()):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f"{row['win%']}% ({int(row['total'])} games)", va="center", fontsize=8)
plt.tight_layout()
plt.show()

## Volume vs win rate — spot the problem openings

In [ ]:
# Openings you play a lot but lose in — highest priority to fix
fig, ax = plt.subplots(figsize=(12, 7))
scatter_data = all_l2[all_l2["total"] >= 5]

ax.scatter(scatter_data["total"], scatter_data["win%"],
           c=scatter_data["win%"], cmap="RdYlGn", vmin=20, vmax=70,
           s=scatter_data["total"] * 3, alpha=0.7, edgecolors="gray", linewidth=0.5)

ax.axhline(50, color="gray", linestyle="--", linewidth=1)
ax.set_xlabel("Games played")
ax.set_ylabel("Win %")
ax.set_title("Opening volume vs win rate — top-left = high priority to fix")

# Label the worst high-volume openings
priority = scatter_data[(scatter_data["win%"] < 45) & (scatter_data["total"] >= 10)]
for name, row in priority.iterrows():
    short = name[:35] + "..." if len(name) > 35 else name
    ax.annotate(short, (row["total"], row["win%"]),
                fontsize=7, ha="left", xytext=(5, 0), textcoords="offset points")

plt.tight_layout()
plt.show()

## Priority practice list

In [ ]:
# Score = games * (50 - win%) — high score = you play it often AND you lose
all_l2["priority_score"] = all_l2["total"] * (50 - all_l2["win%"])
priority_list = all_l2[all_l2["priority_score"] > 0].sort_values("priority_score", ascending=False).head(10)

print("=== TOP OPENINGS TO PRACTICE ===")
print("(Ranked by: how often you play it × how much you lose)\n")
for i, (name, row) in enumerate(priority_list.iterrows(), 1):
    print(f"{i:2}. {name}")
    print(f"    {int(row['total'])} games | {row['win%']}% win | {row['loss%']}% loss\n")